In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model

In [6]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Paths
source = r"C:\Users\HP\Downloads\example\my_photos"
base_dir = r"C:\Users\HP\Downloads\example\data_split"

# Create directories
os.makedirs(f"{base_dir}/train", exist_ok=True)
os.makedirs(f"{base_dir}/val", exist_ok=True)

# Split each class
for class_name in os.listdir(source):
    class_path = os.path.join(source, class_name)
    if not os.path.isdir(class_path):
        continue
    
    # Get all images
    images = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    # 80-20 split
    train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)
    
    # Create class folders
    os.makedirs(f"{base_dir}/train/{class_name}", exist_ok=True)
    os.makedirs(f"{base_dir}/val/{class_name}", exist_ok=True)
    
    # Copy files
    for img in train_imgs:
        shutil.copy(
            os.path.join(class_path, img),
            f"{base_dir}/train/{class_name}/{img}"
        )
    
    for img in val_imgs:
        shutil.copy(
            os.path.join(class_path, img),
            f"{base_dir}/val/{class_name}/{img}"
        )

print("✅ Data split complete!")

✅ Data split complete!


In [7]:
train = train_gen.flow_from_directory(
    r"C:\Users\HP\Downloads\example\data_split\train",
    target_size=(IMG, IMG),
    batch_size=BATCH,
    class_mode="categorical"
)

val = val_gen.flow_from_directory(
    r"C:\Users\HP\Downloads\example\data_split\val",
    target_size=(IMG, IMG),
    batch_size=BATCH,
    class_mode="categorical"
)

Found 2480 images belonging to 53 classes.
Found 627 images belonging to 53 classes.


In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

IMG = 224
BATCH = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)

train = train_gen.flow_from_directory(
    r"C:\Users\HP\Downloads\example\data_split\train",
    target_size=(IMG, IMG),
    batch_size=BATCH,
    class_mode="categorical"
)

val = val_gen.flow_from_directory(
    r"C:\Users\HP\Downloads\example\data_split\val",
    target_size=(IMG, IMG),
    batch_size=BATCH,
    class_mode="categorical"
)

base = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG, IMG, 3)
)

base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation="relu")(x)
out = Dense(train.num_classes, activation="softmax")(x)

model = Model(base.input, out)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(train, validation_data=val, epochs=10)

model.save("food_model.keras")
print("✅ Model trained & saved")

Found 2480 images belonging to 53 classes.
Found 627 images belonging to 53 classes.
Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 146s 2s/step - accuracy: 0.2169 - loss: 3.3035 - val_accuracy: 0.3907 - val_loss: 2.4640
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.4944 - loss: 1.9709 - val_accuracy: 0.5024 - val_loss: 1.9975
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.6218 - loss: 1.4179 - val_accuracy: 0.5327 - val_loss: 1.8235
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.7052 - loss: 1.0948 - val_accuracy: 0.5598 - val_loss: 1.6727
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.7597 - loss: 0.8702 - val_accuracy: 0.5678 - val_loss: 1.6522
Epoch 6/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.8032 - loss: 0.7172 - val_accuracy: 0.5646 - val_loss: 1.6276
Epoch 7/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.8399 - loss: 0.6115 - val_accuracy: 0.5789 - val_loss: 1.6104
Epoch 8/10
78/78 ━━━━━━━━━

In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json

# Recreate the training generator to get class indices
train_gen = ImageDataGenerator(rescale=1./255)
train = train_gen.flow_from_directory(
    r"C:\Users\HP\Downloads\example\data_split\train",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical"
)

# Save class indices to JSON
class_indices = train.class_indices
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=4)

print(f"✅ Saved {len(class_indices)} classes to class_indices.json")
print("\nClass mapping:")
for name, idx in sorted(class_indices.items(), key=lambda x: x[1]):
    print(f"  {idx}: {name}")

Found 2480 images belonging to 53 classes.
✅ Saved 53 classes to class_indices.json

Class mapping:
  0: Blueberry
  1: Gooseberry
  2: Tomato
  3: almond
  4: amaranth leaves
  5: apple
  6: apricot
  7: avocado
  8: banana
  9: beetroot
  10: brinjal
  11: cabbage
  12: capsicum
  13: carrot
  14: cashew(Kaju)
  15: cauliflower
  16: cherry
  17: cluster beans(Guvar)
  18: coconut
  19: cucumber
  20: custard apple
  21: dates
  22: eggs
  23: fig
  24: green chili
  25: guava
  26: kiwi
  27: lettuce
  28: lime
  29: lychee
  30: mango
  31: mint
  32: mustard greens
  33: okra(bhindi)
  34: onion
  35: orange
  36: paneer
  37: papaya
  38: peach
  39: peas
  40: pineapple
  41: pista
  42: plum
  43: pomegranate
  44: potato
  45: radish
  46: spinach
  47: starfruit
  48: strawberry
  49: sweet potato
  50: taro root
  51: walnut
  52: watermelon
